In [1]:
import sys
sys.path.append("../src")

from pyspark.sql import SparkSession
from main import load_trips, load_zones
from m1 import flag_invalid, clean

In [2]:
spark = (
    SparkSession.builder
    .appName("ReportMetrics")
    .master("local[*]")
    .getOrCreate()
)

c:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
print("trips" in globals())
print([x for x in globals() if x in ["spark", "trips", "years", "zones"]])

False
['spark']


In [5]:
years = [2025]
trips = load_trips(
    spark,
    "../databases",
    years
)

In [6]:
flagged = flag_invalid(trips, years)
trips_clean = clean(flagged)

In [7]:
years = [2025]

trips = load_trips(
    spark,
    "../databases",
    years
)

zones = load_zones(
    spark,
    "../databases/taxi_zone_lookup.csv"
)

In [8]:
print("Raw trips:", trips.count())
print("Cleaned trips:", trips_clean.count())
print("Zones:", zones.count())


Raw trips: 19348004
Cleaned trips: 17483653
Zones: 265


In [23]:
import glob
import os

files = glob.glob("../databases/yellow_tripdata_2025-*.parquet")

total_bytes = sum(os.path.getsize(f) for f in files)
total_gb = total_bytes / (1024 ** 3)

print("Files:", len(files))
print(f"Total input size: {total_gb:.2f} GB")

Files: 5
Total input size: 0.31 GB


In [9]:
from m1 import cleaning_stats
stats = cleaning_stats(flagged)
stats.orderBy("year").show(truncate=False)

+----+--------+------------+------------+-----------+------------+------------+--------------+------------+-----------+
|year|total   |removed_fare|removed_dist|removed_pax|removed_time|removed_zone|removed_period|removed_null|removed_any|
+----+--------+------------+------------+-----------+------------+------------+--------------+------------+-----------+
|2007|1       |0           |0           |0          |0           |0           |1             |0           |1          |
|2009|2       |0           |0           |0          |0           |0           |2             |0           |2          |
|2024|21      |0           |0           |0          |0           |0           |21            |0           |21         |
|2025|19347980|1042872     |542643      |111923     |144203      |145768      |0             |0           |1864327    |
+----+--------+------------+------------+-----------+------------+------------+--------------+------------+-----------+



In [10]:
sc = spark.sparkContext
conf = spark.sparkContext.getConf()

print("Spark version:", spark.version)
print("Master:", sc.master)
print("Application name:", sc.appName)
print("Default parallelism:", sc.defaultParallelism)
print("Executor memory:", conf.get("spark.executor.memory", "Not explicitly set"))
print("Driver memory:", conf.get("spark.driver.memory", "Not explicitly set"))
print("Executor cores:", conf.get("spark.executor.cores", "Not explicitly set"))

Spark version: 4.2.0
Master: local[*]
Application name: ReportMetrics
Default parallelism: 16
Executor memory: Not explicitly set
Driver memory: Not explicitly set
Executor cores: Not explicitly set


In [11]:
print("Test:", spark.range(10).count())

Test: 10


In [12]:
import sys
sys.path.append("../src")

import main
import m1
import m2
import m3
import o1
import o2

trips = main.load_trips(spark, "../databases", [2025], 1.0)
zones = main.load_zones(spark, "../databases/taxi_zone_lookup.csv")

print("Raw trips:", trips.count())
print("Zones:", zones.count())

Raw trips: 19348004
Zones: 265


In [ ]:
#M1 performance

import time

start = time.perf_counter()

flagged = m1.flag_invalid(trips, years=[2025])
trips_clean = m1.clean(flagged)

cleaned_count = trips_clean.count()

m1_time = time.perf_counter() - start

print("Cleaned trips:", cleaned_count)
print(f"M1 execution time: {m1_time:.2f} seconds")

Cleaned trips: 17483653
M1 execution time: 3.05 seconds


In [ ]:
#M2 performance

start = time.perf_counter()

demand = m2.build(trips_clean, zones)
m2_rows = demand.count()

m2_time = time.perf_counter() - start

print("M2 output rows:", m2_rows)
print(f"M2 execution time: {m2_time:.2f} seconds")

M2 output rows: 951
M2 execution time: 6.62 seconds


In [15]:
#M3 performance

start = time.perf_counter()

m3_result = m3.build(trips_clean, zones)
m3_rows = m3_result.count()

m3_time = time.perf_counter() - start

print("M3 output rows:", m3_rows)
print(f"M3 execution time: {m3_time:.2f} seconds")

M3 output rows: 1295
M3 execution time: 5.57 seconds


In [16]:
#O1 performance

start = time.perf_counter()

base = o1.addFarePerKm(trips_clean)
stats = o1.bucketStats(base)
flagged_o1 = o1.flagAnomalies(base, stats)
rates = o1.anomaly_rates(flagged_o1, zones)

o1_rows = rates.count()

o1_time = time.perf_counter() - start

print("O1 output rows:", o1_rows)
print(f"O1 execution time: {o1_time:.2f} seconds")

O1 output rows: 58
O1 execution time: 5.66 seconds


In [17]:
#O2 performance

start = time.perf_counter()

flows = o2.build_monthly_flows(trips_clean, zones)
flows_count = flows.count()

top20 = o2.top_routes(flows, n=20)
top20_count = top20.count()

reversals = o2.share_reversals(flows)
reversals_count = reversals.count()

o2_time = time.perf_counter() - start

print("Monthly flows:", flows_count)
print("Top routes:", top20_count)
print("Share reversals:", reversals_count)
print(f"O2 execution time: {o2_time:.2f} seconds")

Monthly flows: 178645
Top routes: 566
Share reversals: 18848
O2 execution time: 51.28 seconds


In [18]:
import os
import platform
import psutil

print("CPU:", platform.processor())
print("Logical processors:", os.cpu_count())
print("Physical cores:", psutil.cpu_count(logical=False))
print("RAM (GB):", round(psutil.virtual_memory().total / (1024**3), 2))
print("Available RAM (GB):", round(psutil.virtual_memory().available / (1024**3), 2))

CPU: AMD64 Family 25 Model 68 Stepping 1, AuthenticAMD
Logical processors: 16
Physical cores: 8
RAM (GB): 15.24
Available RAM (GB): 2.2
